# Cenário do Projeto: Controle de Estoque (Apache Iceberg)

**Descrição do Cenário:** 
O cenário abordado neste projeto simula um sistema simplificado de **Controle de Estoque** de uma loja de eletrônicos. A tabela principal armazena informações vitais dos produtos, como ID, nome do produto, categoria, quantidade disponível e preço unitário. 

**Conjunto de Dados (Dataset):**
O conjunto de dados utilizado foi **criado manualmente (Mock Data)** diretamente via código no Apache Spark para fins de demonstração das operações ACID (Insert, Update, Delete) no formato Apache Iceberg.

**Modelo Entidade-Relacionamento (ER):**
Abaixo está a representação da nossa entidade `estoque_iceberg`.

```mermaid
erDiagram
    ESTOQUE {
        int id PK
        string nome_produto
        string categoria
        int quantidade
        float preco
    }
```

*(Adicione uma imagem ilustrativa do banco de dados na pasta e referencie aqui: `![Modelo ER](./imagem_estoque.png)`)*

In [ ]:
from pyspark.sql import SparkSession
import logging

logging.getLogger("py4j").setLevel(logging.DEBUG)

# Inicializando a SparkSession com as configurações do Apache Iceberg
spark = SparkSession.builder \
  .appName("IcebergLocalDevelopment") \
  .config('spark.jars.packages', 'org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1') \
  .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
  .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
  .config("spark.sql.catalog.local.type", "hadoop") \
  .config("spark.sql.catalog.local.warehouse", "spark-warehouse/iceberg") \
  .getOrCreate()

spark

: 

### DDL - Criação da Tabela
Aqui executamos o comando Data Definition Language (DDL) para criar a estrutura da tabela do nosso estoque. Definimos as colunas iniciais e o formato da tabela indicando `USING iceberg`. Repare que utilizamos o prefixo do catálogo `local.`.

In [ ]:
spark.sql("""
  CREATE TABLE IF NOT EXISTS local.estoque_iceberg (id INT, nome_produto STRING) USING iceberg
""")

spark.sql("SELECT * FROM local.estoque_iceberg").show()

### INSERT - Inserção de Dados
O comando `INSERT INTO` é utilizado para popular a tabela com nossos produtos iniciais. O Iceberg garante que essa inserção seja feita de forma transacional e segura.

In [ ]:
spark.sql("INSERT INTO local.estoque_iceberg VALUES (1, 'Notebook Dell'), (2, 'Mouse Logitech'), (3, 'Teclado Mecânico')")

spark.sql("SELECT * FROM local.estoque_iceberg").show()

### UPDATE - Atualização de Dados
Utilizamos o comando `UPDATE` para alterar registros existentes no estoque. Primeiro utilizamos a funcionalidade de *Schema Evolution* do Iceberg para adicionar novas colunas sem reescrever a tabela e, em seguida, atualizamos os valores a nível de linha.

In [ ]:
# Evolução de Schema (Adicionando novas colunas)
spark.sql("""
    ALTER TABLE local.estoque_iceberg ADD COLUMN categoria STRING, quantidade INT, preco FLOAT
""")

# Executando os UPDATES linha por linha
spark.sql("UPDATE local.estoque_iceberg SET categoria = 'Computadores', quantidade = 15, preco = 3500.00 WHERE id = 1")
spark.sql("UPDATE local.estoque_iceberg SET categoria = 'Periféricos', quantidade = 50, preco = 120.00 WHERE id = 2")
spark.sql("UPDATE local.estoque_iceberg SET categoria = 'Periféricos', quantidade = 30, preco = 250.00 WHERE id = 3")

spark.sql("SELECT * FROM local.estoque_iceberg").show()

### DELETE - Exclusão de Dados
O comando `DELETE FROM` remove um registro específico da tabela. Abaixo, vamos simular que o 'Teclado Mecânico' (id = 3) esgotou ou saiu de linha, deletando-o do estoque.

In [ ]:
spark.sql("DELETE FROM local.estoque_iceberg WHERE id = 3")

spark.sql("SELECT * FROM local.estoque_iceberg").show()

### Snapshots e Histórico (Time Travel)
O Apache Iceberg rastreia cada modificação na tabela através de *Snapshots*. Consultando a tabela de metadados `.snapshots`, podemos ver o histórico de todas as transações (adições, atualizações e exclusões) feitas no nosso estoque.

In [ ]:
from IPython.display import display, HTML
display(HTML("<style>pre { white-space: pre !important; }</style>"))

# Consultando os metadados de snapshots do Iceberg
spark.sql("SELECT committed_at, operation, snapshot_id, summary FROM local.estoque_iceberg.snapshots").show(truncate=False)